![Banner sesión 5](https://github.com/cvail-research/Hands-on-Computer-Vision/blob/main/sesiones/sesion5/banner_sesion_5.png?raw=true)

# <font color="EB9A54"><center> **Hands-on Sesión 5: Estimación pasiva de la profundidad 📷 🤖** </center></font>


# **Contenido**

[**1. Pipeline con modelos fundacionales**](#pipeline)

&emsp;&emsp;[**1.1 Imagen de entrada**](#entrada)

&emsp;&emsp;[**1.2 Estimación monocular de profundidad**](#profundidad)

&emsp;&emsp;[**1.3 Reconstrucción 3D desde profundidad**](#reconstruccion)

&emsp;&emsp;[**1.4 Comparación con ground truth**](#comparacion)

[**2. Reconstrucción 3D directa**](#reconstruccion-directa)


## <font color='#4C5FDA'>**1. Pipeline con modelos fundacionales**</font> <a name="pipeline"></a>

En el notebook anterior construimos profundidad a partir de geometría estéreo clásica. Aquí nos enfocamos en el flujo moderno con modelos fundacionales: una imagen de entrada, un modelo de profundidad monocular y una reconstrucción 3D interactiva.

En esta versión comparamos dos modelos monoculares métricos sobre la misma imagen: **UniK3D** y **Depth Anything V2 Metric Outdoor**. Para que la comparación sea justa, en ambos casos usamos el mapa de profundidad para construir la nube de puntos.

Referencias: [UniK3D](https://github.com/lpiccinelli-eth/UniK3D), [Depth Anything V2 Metric Outdoor](https://huggingface.co/depth-anything/Depth-Anything-V2-Metric-Outdoor-Small-hf).


![Depth Anything V2](https://depth-anything-v2.github.io/static/images/teaser.png)

In [ ]:
# @title **Instalar paquetes**
import importlib.util
import subprocess
import sys
from importlib.metadata import PackageNotFoundError, version

def version_menor(version_actual, version_minima):
    actual = tuple(map(int, version_actual.split(".")[:3]))
    minima = tuple(map(int, version_minima.split(".")[:3]))
    return actual < minima

paquetes = {
    "plotly": "plotly",
    "huggingface_hub": "huggingface_hub",
    "safetensors": "safetensors",
    "timm": "timm",
}

faltantes = [pkg for modulo, pkg in paquetes.items() if importlib.util.find_spec(modulo) is None]
try:
    if version_menor(version("transformers"), "4.45.0"):
        faltantes.append("transformers>=4.45.0")
except PackageNotFoundError:
    faltantes.append("transformers>=4.45.0")

if faltantes:
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", *faltantes])

if importlib.util.find_spec("unik3d") is None:
    subprocess.check_call([
        sys.executable,
        "-m",
        "pip",
        "install",
        "-q",
        "git+https://github.com/lpiccinelli-eth/UniK3D.git",
    ])


In [ ]:
# @title **Imports y funciones auxiliares**
from pathlib import Path
from io import BytesIO
import base64
import cv2
import matplotlib.pyplot as plt
import numpy as np
import plotly.graph_objects as go
import requests
import torch
from PIL import Image
from transformers import AutoImageProcessor, AutoModelForDepthEstimation
from unik3d.models import UniK3D

BASE_URL = "https://raw.githubusercontent.com/cvail-research/Hands-on-Computer-Vision/main/sesiones/sesion5/{}"
ARCHIVOS_SESION = {
    "color.png": BASE_URL.format("color.png"),
    "28.png": BASE_URL.format("28.png"),
}

def resolver_ruta_sesion5(*partes, debe_existir=False):
    cwd = Path.cwd().resolve()
    for base in (cwd, *cwd.parents):
        for raiz in (base, base / "sesiones" / "sesion5"):
            ruta = raiz.joinpath(*partes)
            if ruta.exists():
                return ruta
    ruta = Path.cwd().joinpath(*partes)
    if debe_existir and not ruta.exists():
        raise FileNotFoundError(f"No se encontró el archivo: {ruta}")
    return ruta

def obtener_archivo_sesion(nombre_archivo):
    ruta = resolver_ruta_sesion5(nombre_archivo)
    if ruta.exists():
        return ruta

    url = ARCHIVOS_SESION[nombre_archivo]
    respuesta = requests.get(url)
    if respuesta.status_code != 200:
        raise FileNotFoundError(f"No se encontró {ruta} ni se pudo descargar {url}")

    ruta.write_bytes(respuesta.content)
    return ruta

def mostrar_imagen_y_depth(imagen, depth, titulo_depth="Profundidad"):
    fig, axes = plt.subplots(1, 2, figsize=(12, 5))
    axes[0].imshow(imagen)
    axes[0].set_title("Imagen RGB")
    axes[0].axis("off")

    im = axes[1].imshow(depth, cmap="inferno")
    axes[1].set_title(titulo_depth)
    axes[1].axis("off")
    fig.colorbar(im, ax=axes[1], fraction=0.046, pad=0.04)
    plt.tight_layout()
    plt.show()

def normalizar_depth(depth):
    depth = np.asarray(depth, dtype=np.float32)
    mask = np.isfinite(depth)
    if not mask.any():
        return np.zeros_like(depth)
    d_min, d_max = depth[mask].min(), depth[mask].max()
    if d_max - d_min == 0:
        return np.zeros_like(depth)
    return (depth - d_min) / (d_max - d_min)

def inferir_unik3d(image, model_name="lpiccinelli/unik3d-vits"):
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    model = UniK3D.from_pretrained(model_name).to(device)
    model.eval()

    rgb = torch.from_numpy(np.array(image.convert("RGB"))).permute(2, 0, 1).to(device)
    with torch.no_grad():
        predictions = model.infer(rgb)

    depth = predictions["depth"].detach().cpu().squeeze().numpy()
    points = predictions["points"].detach().cpu().squeeze()
    return depth, points

def inferir_depth_anything_metric(image, model_name="depth-anything/Depth-Anything-V2-Metric-Outdoor-Small-hf"):
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    processor = AutoImageProcessor.from_pretrained(model_name)
    model = AutoModelForDepthEstimation.from_pretrained(model_name).to(device)
    model.eval()

    inputs = processor(images=image, return_tensors="pt").to(device)
    with torch.no_grad():
        outputs = model(**inputs)

    post_processed = processor.post_process_depth_estimation(
        outputs,
        target_sizes=[(image.height, image.width)],
    )
    depth = post_processed[0]["predicted_depth"].detach().cpu().squeeze().numpy()
    return depth

def visualizar_pointcloud_desde_depth(image, depth, max_points=70000, focal_px=None):
    image_np = np.asarray(image.convert("RGB"))
    depth_np = np.asarray(depth, dtype=np.float32)
    h, w = depth_np.shape
    if image_np.shape[:2] != (h, w):
        image_np = np.array(image.resize((w, h), Image.LANCZOS))
    if focal_px is None:
        focal_px = 0.8 * max(h, w)

    yy, xx = np.mgrid[0:h, 0:w]
    z = depth_np
    x = (xx - w / 2) * z / focal_px
    y = (yy - h / 2) * z / focal_px
    points = np.stack([x, y, z], axis=-1).reshape(-1, 3)
    colors = image_np.reshape(-1, 3).astype(np.float32) / 255.0

    mask = np.isfinite(points).all(axis=1) & (points[:, 2] > 0)
    points = points[mask]
    colors = colors[mask]
    if len(points) == 0:
        raise ValueError("No hay puntos válidos para reconstruir.")

    n = min(max_points, len(points))
    idx = np.random.choice(len(points), size=n, replace=False)
    points = points[idx]
    colors = colors[idx]
    colors_rgb = [f"rgb({r},{g},{b})" for r, g, b in (colors * 255).astype(np.uint8)]

    fig = go.Figure(
        data=[
            go.Scatter3d(
                x=points[:, 0],
                y=-points[:, 2],
                z=-points[:, 1],
                mode="markers",
                marker=dict(size=1, color=colors_rgb),
            )
        ],
        layout=dict(
            scene=dict(
                xaxis=dict(visible=True),
                yaxis=dict(visible=True),
                zaxis=dict(visible=True),
                aspectmode="data",
            ),
            margin=dict(l=0, r=0, b=0, t=20),
        ),
    )
    fig.show()
    return points, colors

def visualizar_pointcloud_unik3d_directa(image, points, max_points=70000):
    image_np = np.asarray(image.convert("RGB"))
    points_np = points.detach().cpu().numpy() if hasattr(points, "detach") else np.asarray(points)

    if points_np.ndim == 3 and points_np.shape[0] == 3:
        points_np = np.moveaxis(points_np, 0, -1)
    if points_np.ndim != 3 or points_np.shape[-1] != 3:
        raise ValueError(f"Formato inesperado para points: {points_np.shape}")

    h, w = points_np.shape[:2]
    if image_np.shape[:2] != (h, w):
        image_np = np.array(image.resize((w, h), Image.LANCZOS))

    points_flat = points_np.reshape(-1, 3)
    colors = image_np.reshape(-1, 3).astype(np.float32) / 255.0

    mask = np.isfinite(points_flat).all(axis=1)
    points_flat = points_flat[mask]
    colors = colors[mask]
    if len(points_flat) == 0:
        raise ValueError("No hay puntos válidos para reconstruir.")

    n = min(max_points, len(points_flat))
    idx = np.random.choice(len(points_flat), size=n, replace=False)
    points_flat = points_flat[idx]
    colors = colors[idx]
    colors_rgb = [f"rgb({r},{g},{b})" for r, g, b in (colors * 255).astype(np.uint8)]

    fig = go.Figure(
        data=[
            go.Scatter3d(
                x=points_flat[:, 0],
                y=-points_flat[:, 2],
                z=-points_flat[:, 1],
                mode="markers",
                marker=dict(size=1, color=colors_rgb),
            )
        ],
        layout=dict(
            scene=dict(
                xaxis=dict(visible=True),
                yaxis=dict(visible=True),
                zaxis=dict(visible=True),
                aspectmode="data",
            ),
            margin=dict(l=0, r=0, b=0, t=20),
        ),
    )
    fig.show()
    return points_flat, colors



## <font color='#4C5FDA'>**1.1 Imagen de entrada**</font> <a name="entrada"></a>

Usaremos la imagen reproducible de la sesión para comparar los modelos sobre la misma entrada.


In [ ]:
# @title **Cargar imagen de la sesión**
image_path = obtener_archivo_sesion("color.png")

image = Image.open(image_path).convert("RGB")
plt.figure(figsize=(8, 6))
plt.imshow(image)
plt.title("Imagen de entrada")
plt.axis("off")
plt.show()


## <font color='#4C5FDA'>**1.2 Estimación monocular de profundidad**</font> <a name="profundidad"></a>

Seleccionamos entre UniK3D Small y Depth Anything V2 Metric Outdoor Small. Ambos son públicos y producen profundidad métrica. Usamos la versión outdoor de Depth Anything porque la imagen de referencia corresponde a una escena exterior.


In [ ]:
# @title **Inferencia monocular métrica**
modelo = "UniK3D Small" # @param ["UniK3D Small", "Depth Anything V2 Metric Outdoor Small"]

if modelo == "UniK3D Small":
    predicted_depth, _ = inferir_unik3d(image)
else:
    predicted_depth = inferir_depth_anything_metric(image)

mostrar_imagen_y_depth(image, predicted_depth, f"Profundidad estimada con {modelo}")


## <font color='#4C5FDA'>**1.3 Reconstrucción 3D desde profundidad**</font> <a name="reconstruccion"></a>

Visualizamos la reconstrucción 3D reproyectando el mapa de profundidad del modelo seleccionado con una cámara pinhole aproximada. Así ambos modelos siguen el mismo procedimiento: profundidad → nube de puntos.


In [ ]:
# @title **Nube de puntos desde profundidad**
max_points = 70000 # @param {type:"slider", min:10000, max:150000, step:10000}

points_modelo, colors_modelo = visualizar_pointcloud_desde_depth(
    image,
    predicted_depth,
    max_points=max_points,
)

print(f"Modelo: {modelo}")
print(f"Puntos visualizados: {len(points_modelo)}")


## <font color='#4C5FDA'>**1.4 Comparación con ground truth**</font> <a name="comparacion"></a>

Cuando usamos la imagen de la sesión también tenemos un mapa de profundidad de referencia. Para comparar visualmente, normalizamos ambos mapas a `[0, 1]`.


In [ ]:
# @title **Comparar contra el ground truth de la sesión**
gt_path = obtener_archivo_sesion("28.png")
true_depth = cv2.imread(str(gt_path), cv2.IMREAD_UNCHANGED).astype(np.float32)
if true_depth.ndim == 3:
    true_depth = cv2.cvtColor(true_depth, cv2.COLOR_BGR2GRAY)
true_depth = cv2.resize(true_depth, predicted_depth.shape[::-1], interpolation=cv2.INTER_CUBIC)

pred_norm = normalizar_depth(predicted_depth)
true_norm = normalizar_depth(true_depth)
error_abs = np.abs(pred_norm - true_norm)

fig, axes = plt.subplots(1, 3, figsize=(16, 5))
mapas = [pred_norm, true_norm, error_abs]
titulos = ["Predicción normalizada", "Ground truth normalizado", "Error absoluto"]
cmaps = ["magma", "magma", "viridis"]

for ax, mapa, titulo, cmap in zip(axes, mapas, titulos, cmaps):
    im = ax.imshow(mapa, cmap=cmap)
    ax.set_title(titulo)
    ax.axis("off")
    fig.colorbar(im, ax=ax, fraction=0.046, pad=0.04)

plt.tight_layout()
plt.show()
print(f"MAE normalizado: {np.nanmean(error_abs):.4f}")


## <font color='#4C5FDA'>**2. Reconstrucción 3D directa**</font> <a name="reconstruccion-directa"></a>

UniK3D también predice la geometría 3D de la escena directamente. En esta sección usamos esos puntos 3D sin pasar por nuestra reproyección pinhole. Esto permite comparar dos formas de reconstruir con el mismo modelo: desde profundidad estimada y desde puntos 3D directos.

![Unik3D](https://cdn.bytez.com/mobilePapers/v2/cvpr/32554/images/2-0.png)


In [ ]:
# @title **Reconstrucción directa con UniK3D**
max_points_directa = 70000 # @param {type:"slider", min:10000, max:150000, step:10000}

depth_unik3d_directa, points_unik3d_directa = inferir_unik3d(image)
points_directos, colors_directos = visualizar_pointcloud_unik3d_directa(
    image,
    points_unik3d_directa,
    max_points=max_points_directa,
)

print(f"Puntos visualizados: {len(points_directos)}")


Nota: En contextos realistas UniK3D suele ser más adecuado para reconstrucción porque fue entrenado para predecir geometría 3D métrica de la escena. Depth Anything V2 está enfocado en mapas 2D de profundidad.
